# 04 — Evalvacija klonske ekspanzije (Figure 4)

Oceni napoved klonske ekspanzije za enega heldout pacienta — kateri T-celicni kloni se po imunoterapiji razsirijo.

Primerja dva pristopa (oba kot v originalu, analysis/HNSCC/helpers.py):
- baseline — navaden klasifikator (kNN/NN) na realnih celicah (spodnja meja)
- our (TRIM) — generativna napoved iz naucenega modela

Vhod: holdout{pid}/preds.npz, holdout{pid}/model.pth, data_*.pkl, df_all_tcrs.pkl (s count stolpci iz notebooka 01)
Izhod: ROC AUC + ROC krivulja za vsak pristop

Per-patient (en heldout naenkrat). Za agregat cez vec pacientov razsiri output_folders na seznam map.


## 0. Namestitev in mount

In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
!pip install -q scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Konfiguracija

In [ ]:
# === POTI ===
data_parent_folder = '/content/drive/MyDrive/Diploma/data/processed'
tcr_ae_train_step  = 50100

# === HELDOUT ===
heldout_patient = [24]          # en pacient naenkrat (P24 ima vse 4 kvadrante)
device_str      = 'cuda:0'
seed            = 0      # MORA se ujemati z 03 treningom (bere runs/.../seed{seed}/)

# === ARHITEKTURA (mora se ujemati z notebookom 03) ===
n_channels_base     = 2048
dimz                = 1024
dim_state_embedding = 128

print('Konfiguracija nastavljena. Heldout:', heldout_patient)

## 2. Uvozi in poti

In [ ]:
import numpy as np
import pandas as pd
import os, json, pickle, time
import sklearn.metrics, sklearn.svm, sklearn.neighbors, sklearn.ensemble, sklearn.neural_network
import matplotlib.pyplot as plt
import torch
from torch import nn

tcr_folder    = os.path.join(data_parent_folder, f'tcr_ae/step_{tcr_ae_train_step}')
pca_file      = os.path.join(data_parent_folder, 'data_rna_pca.pkl')
pca_obj_file  = os.path.join(data_parent_folder, 'rna_pca.pkl')
# NOVA HIERARHIJA (ujema se z 03 treningom): runs/P{pid}/seed{s}/rna_tcr/
pid_str = '_'.join(map(str, heldout_patient))
output_folder = os.path.join(data_parent_folder, 'runs', f'P{pid_str}', f'seed{seed}', 'rna_tcr')

device = torch.device(device_str if torch.cuda.is_available() else 'cpu')
np.random.seed(seed); torch.manual_seed(seed)   # reproducibilna stohasticna generacija (reparameterize, np.random.choice)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
fig = plt.figure()
print('Output folder:', output_folder)
print('Device:', device)

## 3. Nalozi podatke

Isti .pkl fajli kot notebook 03. data_labels je DataFrame -> x_label = .values.
df_all_tcrs ima 4 count stolpce ('1'..'4') iz notebooka 01 (P2).

In [ ]:
t = time.time()
# RNA je ze PCA-reduciran (data_rna_pca.pkl iz notebooka 02/03)
with open(pca_file, 'rb') as f:
    data_rna = pickle.load(f)
with open(os.path.join(tcr_folder, 'data_tcr.pkl'), 'rb') as f:
    data_tcr = pickle.load(f)
with open(os.path.join(data_parent_folder, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)
with open(os.path.join(data_parent_folder, 'df_all_tcrs.pkl'), 'rb') as f:
    df_all_tcrs = pickle.load(f)
# PCA objekt (za morebitni inverse_transform pri diffexp; tu ni nujen)
with open(pca_obj_file, 'rb') as f:
    pca = pickle.load(f)

# Stolpci (DataFrame -> pozicije, kot v 03)
col_bloodtumor = data_labels.columns.get_loc('Tissue')
col_prepost    = data_labels.columns.get_loc('Treatment Stage')
col_celltype   = data_labels.columns.get_loc('SubCellType')
col_patient    = data_labels.columns.get_loc('Patient')
col_tcr        = data_labels.columns.get_loc('CDR3(Beta1)')
ID_NULL_TCR    = -1

x_rna   = data_rna
x_tcr   = data_tcr
x_label = data_labels.values

# pri studentu so vse celice TCR-pozitivne (no-op), a ohranimo za zvestobo originalu
mask_preprocess = x_label[:, col_tcr] != ID_NULL_TCR
x_rna   = x_rna[mask_preprocess]
x_tcr   = x_tcr[mask_preprocess]
x_label = x_label[mask_preprocess]

dimrna       = x_rna.shape[-1]
dimtcr       = x_tcr.shape[-1]
num_patients = len(np.unique(data_labels.values[:, col_patient]))

print(f'x_rna {x_rna.shape}, x_tcr {x_tcr.shape}, x_label {x_label.shape}')
print(f'df_all_tcrs {df_all_tcrs.shape} (stolpci {list(df_all_tcrs.columns)})')
print(f'dimrna={dimrna}, dimtcr={dimtcr}, num_patients={num_patients}')
print(f'Nalozeno v {time.time()-t:.1f} s')

## 4. Pomozne funkcije in arhitektura modela

Kopirano iz analysis/HNSCC/helpers.py in notebooka 03 (Generator bere globale).

In [ ]:
def numpy2torch(x, type=torch.FloatTensor):
    return torch.from_numpy(x).type(type).to(device)

def reparameterize(mu, logvar, clamp=5):
    std = logvar.mul(0.5).exp_()
    eps = torch.autograd.Variable(std.data.new(std.size()).normal_())
    return eps.mul(std).add_(mu)

def get_drop_duplicates_mask(x, numpy=True):
    """Iz 2.2.eval_gen.py:533 - odstrani podvojene vektorje (vrne bool mask)."""
    mask, uniques = [], set()
    for row in x:
        if not numpy:
            row = row.detach().cpu().numpy()
        row = tuple(row)
        mask.append(row not in uniques)
        uniques.add(row)
    return mask


class MLP(nn.Module):
    def __init__(self, **kwargs):
        super().__init__()
        nbase = kwargs['nbase']; dim_in = kwargs['dim_in']; dim_out = kwargs['dim_out']
        layers = [1, 2, 4]
        if 'decoder' in kwargs: layers = layers[::-1]
        self.layer1 = nn.Linear(dim_in, nbase // layers[0])
        self.layer2 = nn.Linear(nbase // layers[0], nbase // layers[1])
        self.layer3 = nn.Linear(nbase // layers[1], nbase // layers[2])
        self.out    = nn.Linear(nbase // layers[2], dim_out)
        self.bn1 = nn.BatchNorm1d(nbase // layers[0])
        self.bn2 = nn.BatchNorm1d(nbase // layers[1])
        self.bn3 = nn.BatchNorm1d(nbase // layers[2])
        self.act = kwargs.get('act', torch.nn.LeakyReLU())
    def forward(self, x):
        h1 = self.act(self.bn1(self.layer1(x)))
        h2 = self.act(self.bn2(self.layer2(h1)))
        h3 = self.act(self.bn3(self.layer3(h2)))
        return self.out(h3)


class Generator(nn.Module):
    """Kot notebook 03 (cell-11) - bere globale dimrna/dimtcr/num_patients/dimz/..."""
    def __init__(self):
        super().__init__()
        nbase = n_channels_base
        n_cond = 3 * dim_state_embedding
        self.encoder_rna = MLP(dim_in=dimrna + n_cond, dim_out=dimz * 2, nbase=nbase)
        self.decoder_rna = MLP(dim_in=dimz + n_cond,   dim_out=dimrna,   nbase=nbase, decoder=True)
        self.encoder_tcr = MLP(dim_in=dimtcr + n_cond, dim_out=dimz * 2, nbase=nbase)
        self.decoder_tcr = MLP(dim_in=dimz + n_cond,   dim_out=dimtcr,   nbase=nbase, decoder=True)
        self.register_buffer('bloodtumor_embeddings_matrix', torch.zeros(2, dim_state_embedding))
        self.register_buffer('prepost_embeddings_matrix',    torch.zeros(2, dim_state_embedding))
        self.register_buffer('patient_embeddings_matrix',    torch.zeros(num_patients, dim_state_embedding))
        self.mlp_patient_embeddings_rna = MLP(dim_in=dimrna, dim_out=dim_state_embedding, nbase=nbase)
        self.mlp_patient_embeddings_tcr = MLP(dim_in=dimtcr, dim_out=dim_state_embedding, nbase=nbase)
        self.lrelu = torch.nn.LeakyReLU()
    def forward(self, x, embeddings):
        # OPOMBA: forward se v EVALVACIJI NE uporablja -- eval klice le G.sample()
        # (mu se racuna iz shranjenih recon_*_z v preds.npz, ne prek forward).
        # Ta forward je zato lahko zastarel/2-vejni; ne vpliva na rezultat.
        x_rna = torch.cat([x[0]] + embeddings, axis=-1)
        x_tcr = torch.cat([x[1]] + embeddings, axis=-1)
        z_rna = self.encoder_rna(x_rna); z_tcr = self.encoder_tcr(x_tcr)
        mu     = torch.mean(torch.stack([z_rna[:, :dimz], z_tcr[:, :dimz]]), 0)
        logvar = torch.mean(torch.stack([z_rna[:, dimz:], z_tcr[:, dimz:]]), 0)
        z = reparameterize(mu, logvar)
        shared = torch.cat([z] + embeddings, axis=-1)
        return self.decoder_rna(shared), self.decoder_tcr(shared), [mu, logvar, [z_rna, z_tcr]]
    def sample(self, z, embeddings):
        shared = torch.cat([z] + embeddings, axis=-1)
        return self.decoder_rna(shared), self.decoder_tcr(shared), [None, None, shared]

print('Pomozne funkcije in arhitektura definirane.')

## 5. Nalozi natrenirani model (za TRIM napoved)

Embedding matrike so register_buffer -> nalozijo se iz model.pth.

In [ ]:
G = Generator().to(device)
G.load_state_dict(torch.load(os.path.join(output_folder, 'model.pth'), map_location=device))
G.eval()
print('Model nalozen iz', output_folder)
print('patient_embeddings_matrix:', tuple(G.patient_embeddings_matrix.shape))

## 6. Baseline napoved ekspanzije (per-patient)

Iz helpers.py:284. Ne rabi modela - navaden klasifikator na realnih celicah.
Definicija ekspanzije: klon je vecji v blood-post (stolpec 1) kot blood-pre (stolpec 0).
Per-patient: brez zanke cez vse, fiksiran heldout.

In [ ]:
def baseline_expansion_prediction_single(i_pid, train_on, baseline_model, seed=0, plot=True):
    """Per-patient varianta helpers.py:284. Vrne (roc_auc, real, pred)."""
    np.random.seed(seed); torch.manual_seed(seed)

    # train: blood celice OSTALIH pacientov
    mask_train = np.logical_and(x_label[:, col_patient] != i_pid, x_label[:, col_bloodtumor] == 0)
    labels_masked_train = x_label[mask_train]
    blood_expanded = (df_all_tcrs.iloc[:, 1].fillna(0) > df_all_tcrs.iloc[:, 0].fillna(0)).values
    blood_expanded_train = np.take(blood_expanded, labels_masked_train[:, col_tcr].astype(np.int32))

    # test: blood-pre celice heldout pacienta, ki jih NI v trainu (out-of-distribution)
    mask_test = np.logical_and(x_label[:, col_patient] == i_pid,
               np.logical_and(x_label[:, col_bloodtumor] == 0, x_label[:, col_prepost] == 0))
    train_tcrs = set(x_label[mask_train, col_tcr].tolist())
    mask_test = np.array([m and (x_label[i, col_tcr] not in train_tcrs) for i, m in enumerate(mask_test)])
    labels_masked_test = x_label[mask_test]
    blood_expanded_test = np.take(blood_expanded, labels_masked_test[:, col_tcr].astype(np.int32))
    if blood_expanded_test.sum() == 0:
        print('  ni ekspandiranih test klonov -> preskok'); return None

    # znacilke
    if train_on == 'rna':   X_tr, X_te = x_rna[mask_train], x_rna[mask_test]
    elif train_on == 'tcr': X_tr, X_te = x_tcr[mask_train], x_tcr[mask_test]
    elif train_on == 'both':
        X_tr = np.concatenate([x_rna[mask_train], x_tcr[mask_train]], -1)
        X_te = np.concatenate([x_rna[mask_test],  x_tcr[mask_test]],  -1)
    else: raise Exception('bad train_on')

    # klasifikator + subsample (kot original)
    if baseline_model == 'knn':
        clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=20); r = np.arange(mask_train.sum())
    elif baseline_model == 'NN':
        clf = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=[400,200,100]); r = np.arange(mask_train.sum())
    elif baseline_model == 'svm':
        clf = sklearn.svm.SVC(probability=True); r = np.random.choice(mask_train.sum(), min(10000, mask_train.sum()), replace=False)
    elif baseline_model == 'random_forest':
        clf = sklearn.ensemble.RandomForestClassifier(random_state=0); r = np.arange(mask_train.sum())
    else: raise Exception('bad baseline_model')

    m_tr = get_drop_duplicates_mask(X_tr[r]); m_te = get_drop_duplicates_mask(X_te)
    clf.fit(X_tr[r][m_tr], blood_expanded_train[r][m_tr])
    pred = clf.predict_proba(X_te[m_te])[:, 1]
    real = blood_expanded_test[m_te]

    fpr, tpr, _ = sklearn.metrics.roc_curve(real, pred, pos_label=1)
    roc_auc = sklearn.metrics.auc(fpr, tpr)
    print(f'  baseline {baseline_model}/{train_on}: ROC AUC = {roc_auc:.3f}  (n_test={len(real)}, expanded={int(real.sum())})')
    if plot:
        plt.figure(figsize=(4,4))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
        plt.plot([0,1],[0,1],'--', color='navy', lw=1)
        plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(loc='lower right')
        plt.title(f'Baseline {baseline_model}/{train_on} - P{i_pid}'); plt.show()
    return roc_auc, real, pred


pid = heldout_patient[0]
print(f'Baseline za pacienta {pid}:')
baseline_results = {}
for train_on in ['rna', 'tcr', 'both']:
    for model in ['knn', 'NN']:
        res = baseline_expansion_prediction_single(pid, train_on, model, plot=False)
        if res is not None:
            baseline_results[f'{model}_{train_on}'] = res[0]
print('\nBaseline AUC povzetek:', {k: round(v,3) for k,v in baseline_results.items()})

## 7. TRIM napoved ekspanzije (per-patient)

Iz helpers.py:395. Iz blood-pre celic heldout pacienta vzame z (mu/logvar), generira celice v 4 kondicijah, steje pseudo-klonalnost prek thresh_fitted. Vrne napovedano vs resnicno klonalnost.

tcr_dists iz preds.npz NE beremo (mrtva koda v originalu).

In [ ]:
def our_expansion_prediction_single(output_folder, num_points=10000):
    """Per-patient varianta helpers.py:395. Vrne (real_clon, pseudo_clon) za enega pacienta."""
    with open(os.path.join(output_folder, 'args.txt'), 'r') as f:
        args = json.load(f)
    i_pid = args['heldout_patient'][0] if isinstance(args['heldout_patient'], list) else int(args['heldout_patient'])
    dimz_local = args.get('dimz', dimz)

    with open(os.path.join(output_folder, 'preds.npz'), 'rb') as f:
        npz = np.load(f)
        recon_rna_z = npz['recon_rna_z']
        recon_tcr_z = npz['recon_tcr_z']
        thresh_fitted = float(npz['thresh_fitted'])
        # tcr_dists NAMERNO ne beremo (mrtva koda v originalu)

    # preds.npz vsebuje SAMO heldout celice -> indeksiraj znotraj heldout
    mask_ho = x_label[:, col_patient] == i_pid
    x_label_ho = x_label[mask_ho]
    # blood-pre maska ZNOTRAJ heldout (poravnana z recon_*_z, ki sta shranjena za heldout)
    bb = np.logical_and(x_label_ho[:, col_bloodtumor] == 0, x_label_ho[:, col_prepost] == 0)
    if bb.sum() == 0:
        print(f'  P{i_pid}: ni blood-pre celic -> preskok'); return None

    recon_rna_z = numpy2torch(recon_rna_z)
    recon_tcr_z = numpy2torch(recon_tcr_z)
    mu     = torch.stack([recon_rna_z[bb][:, :dimz_local], recon_tcr_z[bb][:, :dimz_local]], 0).mean(0)
    logvar = torch.stack([recon_rna_z[bb][:, dimz_local:], recon_tcr_z[bb][:, dimz_local:]], 0).mean(0)

    num_samples = max(1, int(num_points / mu.shape[0]))
    print(f'  P{i_pid}: blood-pre celic={mu.shape[0]}, num_samples={num_samples}')

    all_out_tcr = []
    with torch.no_grad():
        for _ in range(num_samples):
            out_tcr_ = []
            z = reparameterize(mu, logvar)
            ones = torch.ones(z.shape[0]).to(device).type(torch.int32)
            for bloodtumor, prepost in [[0,0],[0,1],[1,0],[1,1]]:
                bt  = torch.index_select(G.bloodtumor_embeddings_matrix, 0, bloodtumor * ones)
                pp  = torch.index_select(G.prepost_embeddings_matrix,    0, prepost   * ones)
                pat = torch.index_select(G.patient_embeddings_matrix,    0, i_pid     * ones)
                _, out_tcr, _ = G.sample(z=z, embeddings=[bt, pp, pat])
                out_tcr_.append(out_tcr.detach().cpu().numpy())
            all_out_tcr.append(out_tcr_)
    all_out_tcr = np.concatenate(all_out_tcr, axis=1)   # [4, num_samples*n_bb, dimtcr]

    # pseudo-kloni na kondicijo (thresh_fitted/2, kot original)
    all_pseudo_tcrs = []
    for cond in range(4):
        dists = sklearn.metrics.pairwise_distances(all_out_tcr[cond], all_out_tcr[cond], metric='l1')
        thresh = thresh_fitted / 2
        pt = -10 * np.ones(all_out_tcr[cond].shape[0]); cid = 0
        while (pt == -10).sum() > 0:
            i = np.random.choice(np.argwhere(pt == -10).flatten())
            m = np.logical_and(dists[i] < thresh, pt == -10)
            pt[m] = cid; cid += 1
        all_pseudo_tcrs.append(pt)
    all_pseudo_tcrs = np.stack(all_pseudo_tcrs)
    print('   pseudo-klonska raznolikost/kondicija:',
          [round(len(np.unique(t))/all_pseudo_tcrs.shape[1], 3) for t in all_pseudo_tcrs])

    # klonalnost na celico na kondicijo
    clon = []
    for i in range(all_out_tcr.shape[1]):
        clon.append([(all_pseudo_tcrs[c] == all_pseudo_tcrs[c, i]).sum() for c in range(4)])
    clon = np.array(clon)
    pseudo_clon = np.mean(np.array(np.array_split(clon, num_samples)), 0)   # povprecje cez batche

    # resnicna klonalnost iz df_all_tcrs (4 count stolpci) za blood-pre celice heldout pacienta
    real_clon = np.take(df_all_tcrs.fillna(0).values, x_label_ho[bb, col_tcr].astype(int), axis=0)

    return real_clon, pseudo_clon


print('TRIM napoved za', output_folder)
res = our_expansion_prediction_single(output_folder)
results_real   = [res[0]] if res is not None else []
results_pseudo = [res[1]] if res is not None else []

## 8. Koncna ROC AUC

Iz 2.2.eval_gen.py:569-575:
- x = resnicna ekspanzija (blood-post > blood-pre)
- y = napovedan fold-change pseudo-klonalnosti (post/pre), filtriran

In [ ]:
if len(results_real) > 0:
    all_real   = np.concatenate(results_real,   axis=0)
    all_pseudo = np.concatenate(results_pseudo, axis=0)

    x = (all_real[:, 0] < all_real[:, 1]).astype(int)          # resnica: BA > BB
    with np.errstate(divide='ignore', invalid='ignore'):
        y = all_pseudo[:, 1] / all_pseudo[:, 0]                # napoved: BA/BB
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    y *= (all_pseudo[:, 0] >= 2).astype(int)                   # filter nezanesljivih (BB<2)

    print(f'Tock: {len(x)}, ekspandiranih (resnica): {int(x.sum())}, ne: {int((1-x).sum())}')
    if x.sum() == 0 or x.sum() == len(x):
        print('OPOZORILO: en sam razred -> ROC ni definiran (premalo podatkov za enega pacienta).')
    else:
        fpr, tpr, _ = sklearn.metrics.roc_curve(x, y, pos_label=1)
        roc_auc = sklearn.metrics.auc(fpr, tpr)
        print(f'\n=== TRIM ekspanzija ROC AUC (P{heldout_patient[0]}): {roc_auc:.3f} ===')

        plt.figure(figsize=(5,5))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'TRIM (AUC = {roc_auc:.2f})')
        if 'baseline_results' in dir() and len(baseline_results) > 0:
            best = max(baseline_results, key=baseline_results.get)
            plt.plot([], [], ' ', label=f'najboljsi baseline: {best} = {baseline_results[best]:.2f}')
        plt.plot([0,1],[0,1],'--', color='navy', lw=1)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'Napoved klonske ekspanzije - P{heldout_patient[0]}')
        plt.legend(loc='lower right'); plt.tight_layout()
        plt.savefig(os.path.join(output_folder, 'expansion_roc.png'), dpi=150)
        plt.show()
        print('Shranjeno:', os.path.join(output_folder, 'expansion_roc.png'))
else:
    print('Ni rezultatov (our_expansion_prediction vrnil None).')

## 9. Confusion matrix + per-razred metrike

ROC AUC je ena agregatna stevilka. Confusion matrix pokaze DEJANSKE napovedi vs resnica
po razredih (expanded / not) + delez pravilnosti za vsakega.

Ker je `y` zvezen (fold-change), potrebujemo PRAG za diskretno napoved. Prikazemo pri 3 pragovih:
1. **y > 0** (katera koli ne-nicelna napoved = expanded)
2. **Youden optimum** (najboljsa locnica iz ROC: max(TPR-FPR))
3. **prag za enako bazno stopnjo** (napove ~toliko % expanded kot je v resnici)


In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

def report(x, y, thresh, label):
    pred = (y >= thresh).astype(int) if thresh > 0 else (y > 0).astype(int)
    cm = confusion_matrix(x, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    prec, rec, f1, _ = precision_recall_fscore_support(x, pred, labels=[0,1], zero_division=0)
    print(f"\n=== {label} (prag y {'>' if thresh==0 else '>='} {thresh:.3f}) ===")
    print(f"  Napovedanih EXPANDED:     {int(pred.sum())}/{len(pred)}  ({100*pred.mean():.1f}%)")
    print(f"  Napovedanih NOT expanded: {int((1-pred).sum())}/{len(pred)}  ({100*(1-pred).mean():.1f}%)")
    print(f"  Confusion matrix:")
    print(f"                  napoved:NOT   napoved:EXP")
    print(f"    resnica:NOT      {tn:5d}        {fp:5d}     (specificnost {tn/(tn+fp) if tn+fp else 0:.2f})")
    print(f"    resnica:EXP      {fn:5d}        {tp:5d}     (recall/obcutljivost {tp/(tp+fn) if tp+fn else 0:.2f})")
    print(f"  NOT expanded:  precision={prec[0]:.2f}  recall={rec[0]:.2f}  F1={f1[0]:.2f}")
    print(f"  EXPANDED:      precision={prec[1]:.2f}  recall={rec[1]:.2f}  F1={f1[1]:.2f}")
    return pred

if len(results_real) > 0 and 0 < x.sum() < len(x):
    print(f"Skupaj celic: {len(x)} | resnicno EXPANDED: {int(x.sum())} ({100*x.mean():.1f}%) | NOT: {int((1-x).sum())}")

    # 1) prag y > 0
    report(x, y, 0.0, "PRAG 1: y > 0 (katera koli napoved ekspanzije)")

    # 2) Youden optimum iz ROC
    fpr, tpr, thr = sklearn.metrics.roc_curve(x, y, pos_label=1)
    youden_idx = (tpr - fpr).argmax()
    youden_thresh = thr[youden_idx]
    report(x, y, youden_thresh, "PRAG 2: Youden optimum (max TPR-FPR)")

    # 3) prag za enako bazno stopnjo (napove toliko % kot resnica)
    base_rate = x.mean()
    q_thresh = np.quantile(y, 1 - base_rate)
    report(x, y, max(q_thresh, 1e-9), f"PRAG 3: enaka bazna stopnja ({100*base_rate:.1f}% expanded)")

    print("\n" + "="*60)
    print("INTERPRETACIJA:")
    print("  - Visok recall NOT + nizji recall EXP = model odlicno prepozna")
    print("    samotne klone, del ekspandiranih zgresi (v y=0 mnozici).")
    print("  - To potrjuje: y=0 je vecinoma PRAVILNA napoved (samotni kloni).")
else:
    print("Ni rezultatov ali en sam razred.")


## 10. Razporeditev celic po razponu countov/genov (utemeljitev QC praga)

Diagnostika za QC prag `min_genes=200`. Bere **surove counte** (`data_rna_counts.pkl`, pred
normalizacijo) za VSE celice po metadata-matchingu, in pokaze **koliko celic pade v vsak razpon**
countov/genov (0-10, 10-50, ... 5k+).

Hipoteza: problematicne kaplje (sum) imajo ~malo countov (~2-10), prave T-celice mnogo vec
(1000+). Ce je vmes (50-200) skoraj prazno, je prag 200 varen — locimo sum od pravih celic.

In [ ]:
import numpy as np, pickle, os
import matplotlib.pyplot as plt
from scipy.sparse import issparse, csr_matrix

# Surovi counti PRED normalizacijo (data_rna_counts.pkl iz notebooka 01)
with open(os.path.join(data_parent_folder, 'data_rna_counts.pkl'), 'rb') as f:
    Xc = pickle.load(f)
Xc = Xc if issparse(Xc) else csr_matrix(Xc)

counts_per_cell = np.asarray(Xc.sum(axis=1)).ravel()          # UMI countov na celico
genes_per_cell  = np.asarray((Xc > 0).sum(axis=1)).ravel()    # izrazenih genov na celico
THRESH = 200

# Eksplicitni razponi (bins) — koliko celic v vsakem
edges  = [0, 10, 50, 100, 200, 500, 1000, 2000, 5000, np.inf]
labels = ['0-10', '10-50', '50-100', '100-200', '200-500', '500-1k', '1k-2k', '2k-5k', '5k+']

def bin_counts(data):
    idx = np.digitize(data, edges[1:-1], right=False)   # v kateri kos pade vsaka celica
    return np.array([(idx == i).sum() for i in range(len(labels))])

n_by_counts = bin_counts(counts_per_cell)
n_by_genes  = bin_counts(genes_per_cell)

# --- TABELA ---
print(f"Skupaj celic (po metadata-matchingu): {len(counts_per_cell)}\n")
print(f"{'razpon':>10} | {'st. celic (UMI counti)':>24} | {'st. celic (izrazeni geni)':>26}")
print("-"*66)
for lab, nc, ng in zip(labels, n_by_counts, n_by_genes):
    marker = '  <- prag 200 je tu' if lab == '100-200' else ''
    print(f"{lab:>10} | {nc:>10} ({100*nc/len(counts_per_cell):>5.1f}%)     | {ng:>10} ({100*ng/len(genes_per_cell):>5.1f}%){marker}")

# --- GRAF: stolpicni, st. celic po razponu ---
INK, ACCENT, GRID = '#1a1a2e', '#e4572e', '#e8e8ec'
x = np.arange(len(labels))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, vals, name in [(axes[0], n_by_counts, 'UMI countov'), (axes[1], n_by_genes, 'izrazenih genov')]:
    # razponi pod pragom 200 obarvani drugace (domnevni sum)
    below_thresh = [i for i,l in enumerate(labels) if l in ['0-10','10-50','50-100','100-200']]
    colors = [ACCENT if i in below_thresh else INK for i in range(len(labels))]
    ax.bar(x, vals, color=colors, width=0.72)
    for xi, v in zip(x, vals):
        if v > 0:
            ax.text(xi, v, f'{v}', ha='center', va='bottom', fontsize=8, color=INK)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_xlabel(f'{name} na celico'); ax.set_title(f'St. celic po razponu: {name}', color=INK)
    ax.grid(axis='y', color=GRID, lw=0.8)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
axes[0].set_ylabel('st. celic')
# legenda barv
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=ACCENT, label='pod pragom 200 (domnevni sum)'),
                        Patch(color=INK, label='nad pragom 200 (prave celice)')],
               fontsize=8, loc='upper right')
fig.suptitle('Razporeditev celic po razponu countov/genov (surovi counti)', fontweight='bold')
fig.tight_layout()
fig.savefig(os.path.join(output_folder, 'qc_count_ranges.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- povzetek: je bimodalno? ---
below = genes_per_cell < THRESH
gray  = np.logical_and(genes_per_cell >= 50, genes_per_cell < 200)
print(f"\nPod pragom 200 genov: {int(below.sum())} ({100*below.mean():.1f}%)")
print(f"Siva cona (50-199 genov): {int(gray.sum())} ({100*gray.mean():.1f}%)")
print("  -> BIMODALNO (prag 200 varen)" if gray.mean() < 0.05 else "  -> siva cona ni prazna, previdno")


In [ ]:
# === BELEZENJE REZULTATA (log_run) -> JSON na Drive (results/logs/) ===
import os as _os, json as _json, time as _time, numpy as _np

def _confusion_at(x, y, thresh):
    pred = (y >= thresh).astype(int) if thresh > 0 else (y > 0).astype(int)
    x = _np.asarray(x).astype(int)
    tp=int(((pred==1)&(x==1)).sum()); tn=int(((pred==0)&(x==0)).sum())
    fp=int(((pred==1)&(x==0)).sum()); fn=int(((pred==0)&(x==1)).sum())
    sf=lambda n,d: float(n/d) if d else 0.0
    pe,re_=sf(tp,tp+fp),sf(tp,tp+fn); fe=sf(2*pe*re_,pe+re_)
    pn,rn=sf(tn,tn+fn),sf(tn,tn+fp); fn_=sf(2*pn*rn,pn+rn)
    return {'thresh':float(thresh),'n_pred_expanded':int(pred.sum()),
            'confusion':{'tn':tn,'fp':fp,'fn':fn,'tp':tp},
            'EXPANDED':{'precision':round(pe,4),'recall':round(re_,4),'f1':round(fe,4)},
            'NOT':{'precision':round(pn,4),'recall':round(rn,4),'f1':round(fn_,4)},
            'specificity':round(sf(tn,tn+fp),4)}

def _jsonable(o):
    if isinstance(o,dict): return {str(k):_jsonable(v) for k,v in o.items()}
    if isinstance(o,(list,tuple)): return [_jsonable(v) for v in o]
    if isinstance(o,_np.integer): return int(o)
    if isinstance(o,_np.floating): return float(o)
    if isinstance(o,_np.ndarray): return o.tolist()
    if isinstance(o,_np.bool_): return bool(o)
    return o

def log_run(log_dir, variant, heldout_patient, roc_auc, x, y, timestamp,
            seed=None, baseline_results=None, config=None, thresh_fitted=None, extra=None):
    try:
        _os.makedirs(log_dir, exist_ok=True)
        x=_np.asarray(x); y=_np.asarray(y)
        pid = heldout_patient[0] if isinstance(heldout_patient,(list,tuple)) else int(heldout_patient)
        th={}; npos=int((x==1).sum()); both=0<npos<len(x)
        if both:
            th['y>0']=_confusion_at(x,y,0.0)
            order=_np.argsort(-y); xs=x[order]; P,N=npos,len(x)-npos
            tpr=_np.cumsum(xs==1)/(P or 1); fpr=_np.cumsum(xs==0)/(N or 1)
            j=int(_np.argmax(tpr-fpr)); yt=float(y[order][j])
            th['youden']=_confusion_at(x,y,max(yt,1e-12))
            br=float(x.mean()); qt=float(_np.quantile(y,1-br))
            th['base_rate']=_confusion_at(x,y,max(qt,1e-9))
        rec={'variant':variant,'heldout_patient':pid,'seed':seed,'timestamp':timestamp,
             'roc_auc':round(float(roc_auc),4) if roc_auc is not None else None,
             'n_cells_eval':int(len(x)),'n_expanded_true':npos,
             'expanded_frac':round(float(x.mean()),4) if len(x) else None,
             'both_classes':bool(both),'thresholds':th,
             'baseline_results':_jsonable(baseline_results) if baseline_results else None,
             'thresh_fitted':float(thresh_fitted) if thresh_fitted is not None else None,
             'config':_jsonable(config) if config else None,
             'extra':_jsonable(extra) if extra else None,
             'x_true':[int(v) for v in x.tolist()], 'y_score':[float(v) for v in y.tolist()]}
        ss=f'_seed{seed}' if seed is not None else ''
        path=_os.path.join(log_dir,f'{variant}_P{pid}{ss}_{timestamp}.json')
        _json.dump(_jsonable(rec),open(path,'w',encoding='utf-8'),ensure_ascii=False,indent=2)
        print(f'[log_run] shranjeno -> {path}  (AUC={rec["roc_auc"]}, variant={variant}, P{pid}, seed={seed})')
        return path
    except Exception as e:
        print(f'[log_run] OPOZORILO: belezenje ni uspelo ({e}). Rezultat NI izgubljen.')
        return None

# --- klic: zabelezi ta zagon ---
RUN_TS = _time.strftime('%Y%m%d-%H%M%S')
LOG_DIR = _os.path.join(data_parent_folder, 'results', 'logs')
# variant: nastavi rocno glede na notebook (rna_tcr | flux_tcr | rna_flux_tcr)
VARIANT = "rna_tcr"
SEED = seed if 'seed' in dir() else None
if 'roc_auc' in dir() and 'x' in dir() and 'y' in dir():
    log_run(log_dir=LOG_DIR, variant=VARIANT, heldout_patient=heldout_patient,
            roc_auc=roc_auc, x=x, y=y, timestamp=RUN_TS, seed=SEED,
            baseline_results=baseline_results if 'baseline_results' in dir() else None,
            config=config if 'config' in dir() else None,
            thresh_fitted=thresh_fitted if 'thresh_fitted' in dir() else None,
            extra={'output_folder': output_folder if 'output_folder' in dir() else None})
else:
    print('[log_run] preskoceno: roc_auc/x/y ne obstajajo (verjetno en sam razred / ni rezultatov).')
